In [63]:
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [64]:
seed = 42
root_path = "/home/stefan/kits-locala/algolymp"

# Data setup

In [65]:
df_train = pd.read_csv(f"{root_path}/train_data.csv")
df_test = pd.read_csv(f"{root_path}/test_data.csv")

In [66]:
def evaluate(reg, X_train, y_train, X_val, y_val):
    scores = cross_val_score(
        reg, X_train, y_train, cv=3, n_jobs=-1, scoring="neg_mean_absolute_error"
    )
    cv = -scores.mean() + scores.std()

    reg.fit(X_train, y_train)
    mae = mean_absolute_error(y_val, reg.predict(X_val))
    return cv, mae

# Subtask 1

In [67]:
def extract_features_subtask1(img_id):
    path = os.path.join(f"{root_path}/images", f"{int(img_id):04d}.png")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    blurred = cv2.GaussianBlur(img, (5, 5), 0)

    # 1. Local Minima Detection
    kernel_el = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
    local_min = cv2.erode(blurred, kernel_el)
    minima_mask = ((blurred == local_min) & (blurred < np.median(blurred) - 5)).astype(
        np.uint8
    ) * 255
    minima_mask = cv2.dilate(
        minima_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    )
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(minima_mask)
    m1 = sum(1 for i in range(1, num_labels) if 20 < stats[i, cv2.CC_STAT_AREA] < 800)

    # 2. Blob Detection
    params = cv2.SimpleBlobDetector_Params()
    params.filterByColor, params.blobColor = True, 0
    params.filterByArea, params.minArea, params.maxArea = True, 80, 1200
    params.filterByCircularity, params.minCircularity = True, 0.3
    params.filterByConvexity = False
    detector = cv2.SimpleBlobDetector_create(params)
    m2 = len(detector.detect(cv2.bitwise_not(cv2.GaussianBlur(img, (3, 3), 0))))

    # 3. Hough Circles
    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        1.2,
        18,
        param1=50,
        param2=22,
        minRadius=6,
        maxRadius=22,
    )
    m3 = len(circles[0]) if circles is not None else 0

    # 4. Global Statistics
    sobelx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    texture_var = np.mean(
        cv2.blur((img.astype(float) - blurred.astype(float)) ** 2, (15, 15))
    )

    return [
        m1,
        m2,
        m3,
        np.mean(grad_mag),
        np.sum(grad_mag > 20) / 1000,
        np.sum(img < np.percentile(img, 15)) / 1000,
        texture_var,
    ]

In [68]:
X1 = np.array([extract_features_subtask1(idx) for idx in df_train["id"]])
y1 = df_train["count"].values

X1_train, X1_val, y1_train, y1_val = train_test_split(
    X1, y1, test_size=0.2, random_state=seed
)

In [69]:
rf_count = RandomForestRegressor(n_estimators=1500, max_depth=11, random_state=seed)

evaluate(rf_count, X1_train, y1_train, X1_val, y1_val)

(np.float64(0.971665654905226), 0.870497122526796)

In [70]:
rf_count.fit(X1, y1)
X_test1 = np.array(
    [extract_features_subtask1(idx) for idx in df_test["id"]]
)

subtask1 = np.clip(np.round(rf_count.predict(X_test1)), 3, 8).astype(int)

# Subtask 2

In [71]:
def extract_features_subtask2(img_id):
    path = os.path.join(f"{root_path}/images", f"{int(img_id):04d}.png")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    blurred = cv2.GaussianBlur(cv2.medianBlur(img, 3), (5, 5), 0)

    sobelx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)

    return [
        np.mean(grad_mag),
        np.std(grad_mag),
        np.percentile(grad_mag, 95),
        np.std(laplacian),
        np.percentile(laplacian, 95) - np.percentile(laplacian, 5),
        np.std(img.astype(float)),
        np.percentile(img, 95) - np.percentile(img, 5),
        np.sum(grad_mag > np.mean(grad_mag) + np.std(grad_mag)) / grad_mag.size,
    ]

In [72]:
X2 = np.array([extract_features_subtask2(idx) for idx in df_train["id"]])
y2 = df_train["age"].values

X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2, y2, test_size=0.2, random_state=seed
)

In [73]:
rf_age = RandomForestRegressor(n_estimators=1000, max_depth=15, random_state=seed)

evaluate(rf_age, X2_train, y2_train, X2_val, y2_val)

(np.float64(0.26260223906307656), 0.21519346479116264)

In [74]:
rf_age.fit(X2, y2)
X_test2 = np.array(
    [extract_features_subtask2(idx) for idx in df_test["id"]]
)

subtask2 = np.clip(np.round(rf_age.predict(X_test2), 1), 1.0, 5.0)

# Submission

In [75]:
def build_subtask(subtask_id, answer):
    return pd.DataFrame(
        {
            "datapointID": df_test["id"],
            "subtaskID": subtask_id,
            "answer": answer,
        }
    )


submission = pd.concat(
    [build_subtask(1, subtask1), build_subtask(2, subtask2)], ignore_index=True
)
submission.head()

,datapointID,subtaskID,answer
0,700,1,7.0
1,701,1,4.0
2,702,1,4.0
3,703,1,6.0
4,704,1,3.0


In [76]:
submission.to_csv(f"{root_path}/submission.csv", index=False)